# 00 — Download raw data (API acquisition)

Downloads two datasets directly via their APIs. The query URL *is* the acquisition method —
versioned in this notebook, so it is reproducible. Each fetch is also logged in
`../data/DOWNLOAD_LOG.md` with URL, date, size, and row count, because both sources revise data
retroactively — the code reproduces the *query*, the log pins down the *snapshot*.

**Subset chosen:** Jan–Feb 2025 + Jul–Aug 2025 (winter vs. summer weather contrast; small enough
for a laptop; the exam sheet explicitly allows subsets).

**Column selection:** the 311 dataset has 43 columns; a first attempt at pulling all of them for
the winter window produced a 140+ MB response that dropped mid-transfer (`IncompleteRead`). Since
only ~10 columns are actually needed for this project (see CONTEXT.md §3), the query uses SODA's
`$select` to request only those server-side — smaller, faster, and more reliable downloads, and a
deliberate, documented choice rather than an accident.

Rule: `data/raw/` files are never edited after this notebook writes them.

In [1]:
import sys, io, time, urllib.parse
from datetime import date
import pandas as pd
import requests

print("python", sys.version)
print("pandas", pd.__version__)
print("requests", requests.__version__)

DOWNLOAD_DATE = date.today().isoformat()
print("download date", DOWNLOAD_DATE)

def fetch_csv(url, retries=4, timeout=120):
    """GET a CSV URL into a DataFrame, retrying on transient network/chunked-transfer errors."""
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            return pd.read_csv(io.BytesIO(resp.content), low_memory=False)
        except (requests.exceptions.RequestException, pd.errors.ParserError) as e:
            last_err = e
            print(f"  attempt {attempt}/{retries} failed: {e!r} -- retrying")
            time.sleep(2 * attempt)
    raise last_err

python 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
pandas 2.3.3
requests 2.34.2
download date 2026-07-12


## 1. NYC 311 Service Requests — winter window (Socrata SODA API)

No API key needed. `$where` filters server-side (only the requested window is downloaded); `$select` restricts to the columns this project actually uses.

In [2]:
SELECT_311 = (
    "unique_key,created_date,closed_date,agency,agency_name,complaint_type,"
    "descriptor,borough,incident_zip,latitude,longitude,status"
)

where_winter = "created_date between '2025-01-01T00:00:00' and '2025-02-28T23:59:59'"
url_311_winter = (
    "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?"
    f"$select={urllib.parse.quote(SELECT_311)}"
    f"&$where={urllib.parse.quote(where_winter)}&$limit=1000000&$order=created_date"
)
print(url_311_winter)

df_winter = fetch_csv(url_311_winter)
print(len(df_winter), df_winter["created_date"].min(), df_winter["created_date"].max())
assert len(df_winter) < 1_000_000, "hit $limit -- download was truncated, raise the limit or split the window"

path_311_winter = f"../data/raw/311_2025-01_2025-02_downloaded_{DOWNLOAD_DATE}.csv"
df_winter.to_csv(path_311_winter, index=False)
print("saved", path_311_winter)

https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$select=unique_key%2Ccreated_date%2Cclosed_date%2Cagency%2Cagency_name%2Ccomplaint_type%2Cdescriptor%2Cborough%2Cincident_zip%2Clatitude%2Clongitude%2Cstatus&$where=created_date%20between%20%272025-01-01T00%3A00%3A00%27%20and%20%272025-02-28T23%3A59%3A59%27&$limit=1000000&$order=created_date


603544 2025-01-01T00:00:12.000 2025-02-28T23:59:48.000


saved ../data/raw/311_2025-01_2025-02_downloaded_2026-07-12.csv


## 2. NYC 311 Service Requests — summer window

In [3]:
where_summer = "created_date between '2025-07-01T00:00:00' and '2025-08-31T23:59:59'"
url_311_summer = (
    "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?"
    f"$select={urllib.parse.quote(SELECT_311)}"
    f"&$where={urllib.parse.quote(where_summer)}&$limit=1000000&$order=created_date"
)
print(url_311_summer)

df_summer = fetch_csv(url_311_summer)
print(len(df_summer), df_summer["created_date"].min(), df_summer["created_date"].max())
assert len(df_summer) < 1_000_000, "hit $limit -- download was truncated, raise the limit or split the window"

path_311_summer = f"../data/raw/311_2025-07_2025-08_downloaded_{DOWNLOAD_DATE}.csv"
df_summer.to_csv(path_311_summer, index=False)
print("saved", path_311_summer)

https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$select=unique_key%2Ccreated_date%2Cclosed_date%2Cagency%2Cagency_name%2Ccomplaint_type%2Cdescriptor%2Cborough%2Cincident_zip%2Clatitude%2Clongitude%2Cstatus&$where=created_date%20between%20%272025-07-01T00%3A00%3A00%27%20and%20%272025-08-31T23%3A59%3A59%27&$limit=1000000&$order=created_date


619913 2025-07-01T00:00:23.000 2025-08-31T23:59:56.000


saved ../data/raw/311_2025-07_2025-08_downloaded_2026-07-12.csv


## 3. NOAA daily weather — Central Park station (NCEI Access API)

No token needed. `units=metric` requested explicitly — the documented answer to the units trap
(GHCN raw data stores tenths of °C/mm; other downloads use Fahrenheit/inches).

In [4]:
url_weather = (
    "https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries"
    "&stations=USW00094728&startDate=2025-01-01&endDate=2025-08-31"
    "&dataTypes=TMAX,TMIN,PRCP,SNOW&units=metric&format=csv"
)
print(url_weather)

wx = fetch_csv(url_weather)
print(len(wx), wx["DATE"].min(), wx["DATE"].max())

# sanity check: NYC TMAX should be roughly -15..40 C. If values look like 250, units are wrong (tenths).
print("TMAX range:", wx["TMAX"].min(), "..", wx["TMAX"].max())
assert wx["TMAX"].max() < 50, "TMAX implausible for metric C -- check units= param"

path_weather = f"../data/raw/weather_centralpark_2025_downloaded_{DOWNLOAD_DATE}.csv"
wx.to_csv(path_weather, index=False)
print("saved", path_weather)

https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&stations=USW00094728&startDate=2025-01-01&endDate=2025-08-31&dataTypes=TMAX,TMIN,PRCP,SNOW&units=metric&format=csv


243 2025-01-01 2025-08-31
TMAX range: -7.1 .. 37.2
saved ../data/raw/weather_centralpark_2025_downloaded_2026-07-12.csv


## 4. Next step

Fill `../data/DOWNLOAD_LOG.md` with the printed URLs, download date, file sizes, and row counts
for all three files. Then compute checksums (SHA-256, below) for fixity and back up the raw files.
`data/raw/` is gitignored — the log + this notebook are the reproducibility record that stands in
for it in version control.

In [5]:
import os, hashlib

for p in [path_311_winter, path_311_summer, path_weather]:
    size_kb = os.path.getsize(p) / 1024
    with open(p, "rb") as f:
        sha256 = hashlib.sha256(f.read()).hexdigest()
    print(f"{p}\n  size: {size_kb:.1f} KB\n  sha256: {sha256}")

../data/raw/311_2025-01_2025-02_downloaded_2026-07-12.csv
  size: 113789.4 KB
  sha256: 09c22339dc6936888ec8b984f300fdb23ff458b1867dc8c1cf997debf70b5b58


../data/raw/311_2025-07_2025-08_downloaded_2026-07-12.csv
  size: 115755.0 KB
  sha256: 242d24d25bd36d273b29bfb9a1965d9d0dbedf2d63dff285eea219f1baa1a1a5
../data/raw/weather_centralpark_2025_downloaded_2026-07-12.csv
  size: 9.9 KB
  sha256: 923c5108e6788d1f1e0a9f004a7ed36563cbb5353c4d067f70457e0d103bf15a
